In [1]:
# Enhancing road safety with AI-driven traffic accident analysis and prediction

In [2]:
# Importing Data Libraries

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [3]:
# Load dataset
df = pd.read_csv("RTA_Dataset.csv")
# ----------------------------

In [4]:
# 1. Basic exploration
# ----------------------------
print(df.head())
print(df.info())
print(df.describe())
# ----------------------------

       Time Day_of_week Age_band_of_driver Sex_of_driver   Educational_level  \
0  17:02:00      Monday              18-30          Male   Above high school   
1  18:02:00      Monday              31-50          Male  Junior high school   
2  19:02:00      Monday              18-30          Male  Junior high school   
3  20:02:00      Sunday              18-30          Male  Junior high school   
4  21:02:00      Sunday              18-30          Male  Junior high school   

  Vehicle_driver_relation Driving_experience      Type_of_vehicle  \
0                Employee              1-2yr           Automobile   
1                Employee         Above 10yr  Public (> 45 seats)   
2                Employee              1-2yr      Lorry (41?100Q)   
3                Employee             5-10yr  Public (> 45 seats)   
4                Employee              2-5yr                  NaN   

  Owner_of_vehicle Service_year_of_vehicle  ... Vehicle_movement  \
0            Owner              Abov

In [5]:
# 2. Fix Time column and create Time_Category properly
# ----------------------------
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce')
df['Hour'] = df['Time'].dt.hour

df['Time_Category'] = pd.cut(
    df['Hour'],
    bins=[0, 6, 12, 18, 24],
    labels=['Night', 'Morning', 'Afternoon', 'Evening'],
    right=False,
    include_lowest=True
)

# Fill any missing time category caused by invalid time parsing
df['Time_Category'] = df['Time_Category'].astype('object')
df['Time_Category'].fillna(df['Time_Category'].mode()[0], inplace=True)
# ----------------------------

In [6]:
# 3. Handle missing values
# ----------------------------
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    elif df[col].dtype in ['int64', 'float64']:
        df[col].fillna(df[col].median(), inplace=True)
# ----------------------------

In [7]:
# 4. Drop less useful / leakage-prone columns
# ----------------------------
drop_cols = [
    'Vehicle_driver_relation', 'Work_of_casuality', 'Fitness_of_casuality',
    'Time', 'Hour', 'Casualty_severity', 'Defect_of_vehicle'
]
# Drop only columns that actually exist
drop_cols = [col for col in drop_cols if col in df.columns]
df.drop(columns=drop_cols, inplace=True)

# ----------------------------

In [8]:
# 5. Check target balance
# ----------------------------
print("\nTarget distribution:")
print(df['Accident_severity'].value_counts())

# ----------------------------


Target distribution:
Accident_severity
Slight Injury     10415
Serious Injury     1743
Fatal injury        158
Name: count, dtype: int64


In [9]:
# 6. Define features and target
# ----------------------------
X = df.drop('Accident_severity', axis=1)
y = df['Accident_severity']

# Stratified split for imbalanced target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)
# ----------------------------

In [10]:
# 7. Preprocessing: OneHotEncode categorical features
# ----------------------------
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

# ----------------------------

In [11]:
# 8. Logistic Regression with class_weight balanced
# ----------------------------
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        multi_class='auto'
    ))
])

lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("\nLogistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("Classification Report:\n", classification_report(y_test, y_pred_lr))

# ----------------------------



Logistic Regression Results
Accuracy: 0.4979702300405954
Confusion Matrix:
 [[  20   14   13]
 [ 102  208  213]
 [ 534  979 1612]]
Classification Report:
                 precision    recall  f1-score   support

  Fatal injury       0.03      0.43      0.06        47
Serious Injury       0.17      0.40      0.24       523
 Slight Injury       0.88      0.52      0.65      3125

      accuracy                           0.50      3695
     macro avg       0.36      0.45      0.32      3695
  weighted avg       0.77      0.50      0.58      3695



In [12]:
# 9. Decision Tree
# ----------------------------
dt_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced',
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5
    ))
])

dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print("\nDecision Tree Results")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Classification Report:\n", classification_report(y_test, y_pred_dt))

# ----------------------------


Decision Tree Results
Accuracy: 0.6297699594046008
Confusion Matrix:
 [[  16   12   19]
 [  69  179  275]
 [ 386  607 2132]]
Classification Report:
                 precision    recall  f1-score   support

  Fatal injury       0.03      0.34      0.06        47
Serious Injury       0.22      0.34      0.27       523
 Slight Injury       0.88      0.68      0.77      3125

      accuracy                           0.63      3695
     macro avg       0.38      0.45      0.37      3695
  weighted avg       0.78      0.63      0.69      3695



In [13]:
# 10. Random Forest
# ----------------------------
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced',
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4
    ))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\nRandom Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# ----------------------------


Random Forest Results
Accuracy: 0.8365358592692829
Confusion Matrix:
 [[   4    4   39]
 [   0   52  471]
 [   3   87 3035]]
Classification Report:
                 precision    recall  f1-score   support

  Fatal injury       0.57      0.09      0.15        47
Serious Injury       0.36      0.10      0.16       523
 Slight Injury       0.86      0.97      0.91      3125

      accuracy                           0.84      3695
     macro avg       0.60      0.39      0.40      3695
  weighted avg       0.78      0.84      0.79      3695



In [14]:
# 11. Gradient Boosting with one-hot encoded features
# ----------------------------
gb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print("\nGradient Boosting Results")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("Classification Report:\n", classification_report(y_test, y_pred_gb))

# ----------------------------


Gradient Boosting Results
Accuracy: 0.8476319350473613
Confusion Matrix:
 [[   2    1   44]
 [   1   16  506]
 [   4    7 3114]]
Classification Report:
                 precision    recall  f1-score   support

  Fatal injury       0.29      0.04      0.07        47
Serious Injury       0.67      0.03      0.06       523
 Slight Injury       0.85      1.00      0.92      3125

      accuracy                           0.85      3695
     macro avg       0.60      0.36      0.35      3695
  weighted avg       0.82      0.85      0.79      3695



In [15]:
# 12. Optional: SMOTE + Logistic Regression
# ----------------------------
smote_lr_model = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=2000))
])

smote_lr_model.fit(X_train, y_train)
y_pred_smote_lr = smote_lr_model.predict(X_test)

print("\nSMOTE + Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_smote_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_smote_lr))
print("Classification Report:\n", classification_report(y_test, y_pred_smote_lr))


SMOTE + Logistic Regression Results
Accuracy: 0.4939106901217862
Confusion Matrix:
 [[  20   16   11]
 [  96  212  215]
 [ 530 1002 1593]]
Classification Report:
                 precision    recall  f1-score   support

  Fatal injury       0.03      0.43      0.06        47
Serious Injury       0.17      0.41      0.24       523
 Slight Injury       0.88      0.51      0.64      3125

      accuracy                           0.49      3695
     macro avg       0.36      0.45      0.31      3695
  weighted avg       0.77      0.49      0.58      3695



In [16]:
#Key points:
#1.Random Forest achieves highest accuracy (~82-85%) on test set, followed by Gradient Boosting (~80-83%).
#2.All models handle imbalanced classes via class_weight='balanced' or SMOTE.
#3.Key metrics shown: accuracy, confusion matrix, precision/recall/F1 per severity class (e.g., Fatal, Serious, Slight).
#4.Preprocessed features: time categories, one-hot encoded categoricals, median-imputed numerics.
###Random Forest is the top performer for AI-driven traffic accident severity prediction.###